In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# LTAU 하이퍼파라미터(ltau_bins, ltau_k) 12개 조합 간 fold별 성능 차이가
# 통계적으로 유의미한지 ANOVA로 검정한다.
# 참고: run_cue_ltau_param_1.sh / run_cue_ltau_param_2.sh
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# run_cue_ltau_param_1.sh / _2.sh의 LTAU_BINS_LIST / LTAU_K_LIST와 동일하게 맞춘다.
LTAU_BINS_LIST = [1250, 2500, 5000, 10000]
LTAU_K_LIST = [3, 5, 7]

In [2]:
def load_fold_metrics(dataset, target, bins, k, task_type='regression'):
    """predictions_all.csv(5-fold 예측 전체)를 읽어 fold별 rmse/r2/pcc를 계산한다.

    주의: train.py::aggregate_metrics()의 fold 루프(정렬된 Fold 순서, GroundTruth/예측값에
    대한 동일한 isfinite 마스킹, 동일한 get_metrics 호출)를 한 글자도 다르지 않게 재현해야
    fold별 값의 평균이 기존 metrics.csv의 rmse/r2/pcc와 소수점까지 정확히 일치한다.
    """
    model_dir = os.path.join(cue_base, dataset, target, 'v1', f'LTAU_BIN{bins}_K{k}')
    path = os.path.join(model_dir, 'predictions_all.csv')
    if not os.path.exists(path):
        return None

    df_ys = pd.read_csv(path)
    folds = sorted(df_ys['Fold'].unique())
    model_name = 'mlp'

    rows = []
    for fold in folds:
        fd = df_ys[df_ys['Fold'] == fold]

        y_true = fd['GroundTruth'].values.astype(float)
        y_pred = fd[model_name].values.astype(float)

        mask = np.isfinite(y_true) & np.isfinite(y_pred)
        y_true_f = y_true[mask]
        y_pred_f = y_pred[mask]

        if y_true_f.size == 0:
            continue

        met = get_metrics(y_true_f, y_pred_f, task_type=task_type)
        rows.append({'Fold': fold, 'rmse': met['rmse'], 'r2': met['r2'], 'pcc': met['pcc']})

    return pd.DataFrame(rows) if rows else None

import sys
sys.path.append('/data/rlawlsgurjh/work/gitlab/CUE/cmfe')
from cue.utils.metric import get_metrics

raw_rows = []
for d_name in datasets:
    for target in targets:
        for bins in LTAU_BINS_LIST:
            for k in LTAU_K_LIST:
                df_fold = load_fold_metrics(d_name, target, bins, k)
                if df_fold is None:
                    continue
                for _, r in df_fold.iterrows():
                    raw_rows.append({
                        'Dataset': d_name, 'Target': target, 'BIN': bins, 'K': k, 'Fold': r['Fold'],
                        'rmse': r['rmse'], 'r2': r['r2'], 'pcc': r['pcc'],
                    })

df_folds = pd.DataFrame(raw_rows)
print(f"✅ Fold별 데이터 로드 완료: {len(df_folds)}개의 행이 수집되었습니다 (기대값: 2 x 4 x 12 x 5 = 480).")

✅ Fold별 데이터 로드 완료: 480개의 행이 수집되었습니다 (기대값: 2 x 4 x 12 x 5 = 480).


In [3]:
def plot_fold_boxplot(df_folds, dataset, target_name, metric_name, output_dir):
    """지정한 dataset/target/metric에 대해 (BIN, K) 조합별 fold 값(5개씩) 분포를 box plot으로 표시한다.
    (개별 fold 값은 stripplot으로 함께 겹쳐 표시해 표본 수를 눈으로 확인할 수 있게 한다.)
    조합이 12개로 많아 범례 대신 x축에 BIN_K 라벨을 회전 표시한다."""
    df_sub = df_folds[(df_folds['Dataset'] == dataset) & (df_folds['Target'] == target_name)]
    if df_sub.empty or metric_name not in df_sub.columns:
        return

    df_sub = df_sub.copy()
    df_sub['Combo'] = df_sub['BIN'].astype(str) + '_' + df_sub['K'].astype(str)
    combo_order = [f'{bins}_{k}' for bins in LTAU_BINS_LIST for k in LTAU_K_LIST]

    fig, ax = plt.subplots(figsize=(11, 4.5))

    sns.boxplot(
        data=df_sub, x='Combo', y=metric_name, order=combo_order, hue='Combo',
        palette='viridis', ax=ax, width=0.6, showfliers=False, legend=False,
    )
    sns.stripplot(
        data=df_sub, x='Combo', y=metric_name, order=combo_order,
        color='black', alpha=0.7, size=4, ax=ax,
    )

    ax.set_title(f"{dataset} / {target_name} / {metric_name}")
    ax.set_xlabel('LTAU_BINS_K')
    ax.set_ylabel(metric_name.upper())
    ax.tick_params(axis='x', labelrotation=45)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [4]:
plot_base_dir = os.path.join(cue_base, 'plots', 'ltau_param_anova')

for d_name in datasets:
    for target in targets:
        g_name = 'DIS' if d_name == 'DIS' else 'INH'
        output_path = os.path.join(plot_base_dir, g_name)
        for m in metrics_of_interest:
            plot_fold_boxplot(df_folds, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/DIS/PI_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/DIS/PI_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/DIS/PI_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/DIS/Ssel_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/DIS/Ssel_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/DIS/Ssel_pcc.png


... (14 outputs left out)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/INH/WS(alpha=2)_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/INH/RS(k=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/INH/RS(k=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_anova/INH/RS(k=2)_pcc.png
